# Phase 5: Mesh Reconstruction (Poisson Surface Reconstruction)

**Pipeline position:**
```
Phase 1–4: Data, Features, SfM, MVS  ✓
Phase 5: Mesh Reconstruction          ← YOU ARE HERE
Phase 6: Texture Mapping
...
```

---

## Why Convert a Point Cloud to a Mesh?

The dense point cloud from Phase 4 is a set of unordered 3-D samples.  A raw point cloud:
- Has **no connectivity** — there is no notion of which points form a surface
- **Cannot be rendered** efficiently in real-time (no rasterisation)
- **Cannot measure volume** or surface area
- **Cannot be textured** (texture coordinates require a parameterisable surface)

A **triangle mesh** solves all of these.  It explicitly represents the surface as a set of triangles — each triangle has a well-defined normal, area, and can receive UV texture coordinates.

---

## Poisson Surface Reconstruction

Poisson (Kazhdan et al., 2006) is the standard algorithm for high-quality surface reconstruction from point clouds with oriented normals.

### The Core Idea: Indicator Function

Define an **indicator function** $\chi(\mathbf{x})$:
$$\chi(\mathbf{x}) = \begin{cases} 1 & \mathbf{x} \text{ is inside the solid} \\ 0 & \mathbf{x} \text{ is outside} \end{cases}$$

The gradient $\nabla \chi$ is zero everywhere except at the surface, where it equals the surface normal.  This gives us the Poisson equation:

$$\nabla^2 \chi = \nabla \cdot \mathbf{V}$$

where $\mathbf{V}$ is a vector field built from the oriented normals of the input points.  Solving this PDE gives $\chi$, and the surface is extracted as the **isosurface** $\chi = 0.5$.

### Why Poisson is Good for Photogrammetry

1. **Global** — it considers all points simultaneously, producing a smooth, watertight surface without holes
2. **Robust to noise** — the PDE formulation averages out small errors in point positions
3. **Watertight** — the output is always a closed manifold (useful for volume computation and rendering)

### The `depth` Parameter

Controls the octree resolution:
- `depth=6` → coarse, misses fine detail
- `depth=8` → good quality (recommended, used here)
- `depth=10` → fine detail, high memory and compute cost

Octree cell size ≈ scene_size / $2^{\text{depth}}$.  At depth=8 and scene size 100 m: $100/256 \approx 0.4$ m resolution.

---

## Why Oriented Normals Are Critical

Poisson needs normals that **consistently point outward** from the surface.  A normal pointing inward inverts the indicator function — the reconstruction would create a hole where there should be solid geometry.

For aerial photogrammetry, surfaces are mostly horizontal (ground, rooftops) and the camera is above.  We enforce that normals with a downward Z component are flipped upward — a simple heuristic that works well for urban scenes.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
%matplotlib inline

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# Load the dense cloud produced in Phase 4.
# Try the full-pipeline cloud first; fall back to the 4-camera subset.
dense_path = os.path.join('..', 'outputs', 'dense_cloud.ply')
if not os.path.exists(dense_path):
    dense_path = os.path.join('..', 'outputs', 'dense_cloud_subset.ply')
assert os.path.exists(dense_path), "Run phase4_mvs.ipynb first to create the dense cloud."

pcd = o3d.io.read_point_cloud(dense_path)
pts = np.asarray(pcd.points)
print(f"Dense cloud   : {len(pts):,} points")
print(f"Has colours   : {pcd.has_colors()}")
print(f"Extent X      : {pts[:,0].min():.1f} – {pts[:,0].max():.1f} m")
print(f"Extent Y      : {pts[:,1].min():.1f} – {pts[:,1].max():.1f} m")
print(f"Extent Z      : {pts[:,2].min():.1f} – {pts[:,2].max():.1f} m")

AssertionError: Run phase4_mvs.ipynb first to create the dense cloud.

---

## Step 1 — Normal Estimation

We estimate normals using PCA in a local neighbourhood:
1. For each point, find its $k$ nearest neighbours
2. Fit a plane to these $k$ points by PCA (smallest eigenvalue direction = normal)
3. Orient the normal to be consistent with the neighbours

**Parameters:**
- `radius=2.0` m — neighbourhood radius.  Too small → noisy normals.  Too large → oversmoothed.
- `max_nn=30` — cap on neighbours (for speed)

After estimation, we flip all downward-pointing normals.  In aerial scenes, downward normals usually mean the algorithm chose the wrong orientation for a horizontal surface.

In [ ]:
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=2.0, max_nn=30)
)
pcd.orient_normals_consistent_tangent_plane(k=15)

normals = np.asarray(pcd.normals)
downward = normals[:, 2] < 0
normals[downward] *= -1
pcd.normals = o3d.utility.Vector3dVector(normals)

upward_pct = (normals[:, 2] > 0).mean() * 100
print(f"Normals estimated for {len(normals):,} points")
print(f"Upward normals (Z > 0): {upward_pct:.1f}%")
print(f"Downward normals flipped: {downward.sum():,} points")
print()
print("A high upward % is expected for aerial data (mostly rooftops and ground).")

In [ ]:
# Visualise normal Z-components: 1 = pointing straight up, 0 = horizontal, -1 = down
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(normals[:, 2], bins=60, color='steelblue', edgecolor='none')
axes[0].axvline(0, color='red', linestyle='--', label='Z=0 (horizontal)')
axes[0].set_xlabel('Normal Z component')
axes[0].set_ylabel('Count')
axes[0].set_title('Normal Orientation Distribution')
axes[0].legend()

pts_sub = pts[::10]
nrm_sub = normals[::10]
axes[1].scatter(pts_sub[:,0], pts_sub[:,1], s=0.3,
                c=nrm_sub[:,2], cmap='RdYlGn', vmin=-1, vmax=1, alpha=0.5)
axes[1].set_title('Normal Z (top view) — green=up, red=down')
axes[1].set_xlabel('X (m)'); axes[1].set_ylabel('Y (m)')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

---

## Step 2 — Poisson Surface Reconstruction

The reconstruction produces a mesh **and** a per-vertex density value.  Density measures how many input points support each vertex — low-density vertices are extrapolated (the algorithm invented geometry where there are no point cloud samples).  We prune these.

In [ ]:
from src.mesh.reconstruction import reconstruct_poisson

print("Running Poisson reconstruction (depth=8)...")
mesh_raw = reconstruct_poisson(pcd, depth=8, density_threshold=0.0)  # keep all first
print(f"Raw Poisson mesh: {len(mesh_raw.vertices):,} vertices, {len(mesh_raw.triangles):,} triangles")

In [ ]:
# Re-run to access densities directly for visualisation
mesh_for_density, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=8, linear_fit=False
)
densities = np.asarray(densities)

threshold_pct = 2   # prune bottom 2% by density
threshold_val = np.quantile(densities, threshold_pct / 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(densities, bins=80, color='steelblue', edgecolor='none')
axes[0].axvline(threshold_val, color='red', linestyle='--',
                label=f'{threshold_pct}% threshold = {threshold_val:.2f}')
axes[0].set_xlabel('Vertex density (point support)')
axes[0].set_ylabel('Count')
axes[0].set_title('Poisson Vertex Density Distribution')
axes[0].legend()

# Colour mesh by density to visualise low-support regions
verts = np.asarray(mesh_for_density.vertices)
axes[1].scatter(verts[:,0], verts[:,1], s=0.2,
                c=densities, cmap='RdYlGn', alpha=0.5)
axes[1].set_title('Vertex density (top view)\nred = low support = likely extrapolated')
axes[1].set_xlabel('X (m)'); axes[1].set_ylabel('Y (m)')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

pruned = (densities < threshold_val).sum()
print(f"Vertices to prune: {pruned:,} / {len(densities):,} ({100*pruned/len(densities):.1f}%)")

---

## Step 3 — Post-Processing

Three operations clean the raw Poisson mesh:

1. **Crop to scene bounds** — Poisson reconstructs a closed surface even outside the point cloud region (extrapolated geometry).  We clip everything outside the scene + 5 m margin.

2. **Clean** — remove degenerate triangles (zero area, duplicate vertices, non-manifold edges).  These can cause problems in UV unwrapping and rendering.

3. **Simplify (quadric decimation)** — reduce from potentially millions of faces to a target of 80 000.  Quadric decimation collapses edges while minimising the change to the surface shape.  80 K faces is a good trade-off: enough detail for clean texture mapping, small enough for real-time rendering.

In [ ]:
from src.mesh.reconstruction import reconstruct_poisson
from src.mesh.processing import crop_to_scene, simplify_mesh, clean_mesh

mesh = reconstruct_poisson(pcd, depth=8, density_threshold=0.02)
print(f"After density pruning : {len(mesh.triangles):,} faces")

# Derive scene bounds from the input point cloud.
# This works for both synthetic and real data — no dependency on synthetic_scene module.
pts_arr = np.asarray(pcd.points)
min_b   = pts_arr.min(axis=0)
max_b   = pts_arr.max(axis=0)
print(f"Scene bounds (from cloud): {min_b.round(1)} → {max_b.round(1)} m")
print(f"Crop margin: ±5 m to include Poisson boundary geometry")

mesh = crop_to_scene(mesh, min_b - 5, max_b + 5)
print(f"After scene crop      : {len(mesh.triangles):,} faces")

mesh = clean_mesh(mesh)
print(f"After cleaning        : {len(mesh.triangles):,} faces")

mesh = simplify_mesh(mesh, target_faces=80_000)
mesh.compute_vertex_normals()
print(f"After simplification  : {len(mesh.triangles):,} faces")
print(f"Watertight            : {mesh.is_watertight()}")

In [ ]:
verts = np.asarray(mesh.vertices)
tris  = np.asarray(mesh.triangles)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Top view coloured by height
sc = axes[0].scatter(verts[:,0], verts[:,1], s=0.2,
                     c=verts[:,2], cmap='terrain', alpha=0.6)
plt.colorbar(sc, ax=axes[0], label='Z / height (m)')
axes[0].set_title(f'Mesh top view — {len(tris):,} triangles\ncoloured by height')
axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')
axes[0].set_aspect('equal')

# Side view
axes[1].scatter(verts[:,0], verts[:,2], s=0.2, c=verts[:,2], cmap='terrain', alpha=0.6)
axes[1].set_title('Mesh side view (X-Z)')
axes[1].set_xlabel('X (m)'); axes[1].set_ylabel('Z (m)')

plt.suptitle('Poisson Reconstructed Mesh', fontsize=12)
plt.tight_layout()
plt.show()

# Height histogram — should show a bimodal distribution: ground (~0) and rooftops (5-30m)
plt.figure(figsize=(8, 3))
plt.hist(verts[:,2], bins=80, color='steelblue', edgecolor='none')
plt.xlabel('Vertex height Z (m)')
plt.ylabel('Count')
plt.title('Height Distribution — should show ground + rooftop peaks')
plt.tight_layout()
plt.show()

In [ ]:
out_path = os.path.join('..', 'outputs', 'mesh.ply')
o3d.io.write_triangle_mesh(out_path, mesh)
print(f"Mesh saved: {out_path}")

---

## Summary

| Step | Purpose | Output |
|---|---|---|
| Normal estimation | Required by Poisson (PCA + KNN) | Oriented normals |
| Normal flipping | Ensure upward orientation for aerial data | Consistent normals |
| Poisson (depth=8) | Implicit surface from oriented cloud | Watertight mesh |
| Density pruning | Remove extrapolated geometry | Smaller, cleaner mesh |
| Crop | Remove reconstruction outside scene | Scene-bounded mesh |
| Clean | Remove degenerate elements | Valid manifold |
| Simplify (80K) | Reduce for rendering/texturing | Compact mesh |

### Questions to Think About

- The height histogram should show two peaks: ground (Z ≈ 0) and rooftops (5–30 m).  If the reconstruction failed, what would you see instead?
- Poisson always produces a **watertight** mesh.  Is that always desirable?  (Consider a building with no roof in the point cloud.)
- Why do we simplify to 80 000 faces rather than keeping all faces?  What is the cost of having too many faces in Phase 6 (UV unwrapping)?
- The density threshold prunes the bottom 2% of vertices.  What does a density of 0 mean physically?  (Hint: think about where the point cloud has gaps.)

---

**Next → [Phase 6: Texture Mapping](phase6_texture.ipynb)**